# LLM 입력을 위한 최종 프롬프트 생성 노트북 (v4 - 최종판)

**수정 사항:**
- **(중요)** 일부 제품 카테고리에서 시장 정보가 누락되던 문제를 **완전히 해결**했습니다.
- 추측에 기반한 키워드 매칭 대신, 모든 제품 카테고리를 5개 대표 카테고리에 1:1로 명시적으로 매핑하여 100% 정확도를 보장합니다.

In [ ]:
import json
import pandas as pd

# --- 파일 경로 설정 ---
PRODUCT_FILE = 'product_info_preprocessed.jsonl'
PERSONA_FILE = 'persona_attributes_weighted.jsonl'
COMPETITOR_FILE_CLEANED = 'competitor_prices_cleaned_final.csv'
OUTPUT_FILE = 'prompts_for_llm.jsonl'

## 1. 모든 데이터 불러오기

In [19]:
# 제품 데이터
with open(PRODUCT_FILE, 'r', encoding='utf-8') as f:
    products = [json.loads(line) for line in f]
print(f"제품 {len(products)}개 불러오기 완료.")

# 페르소나 데이터
with open(PERSONA_FILE, 'r', encoding='utf-8') as f:
    personas = [json.loads(line) for line in f]
print(f"페르소나 {len(personas)}개 불러오기 완료.")

# 경쟁사 시장 데이터
try:
    df_competitor = pd.read_csv(COMPETITOR_FILE_CLEANED)
    market_context = {}
    for category, group in df_competitor.groupby('category'):
        min_price = group['price_per_100g'].min()
        max_price = group['price_per_100g'].max()
        market_context[category] = {
            "competitor_price_range_per_100g": f"{int(min_price):,}원 ~ {int(max_price):,}원"
        }
    print(f"경쟁사 시장 데이터 불러오기 완료. (카테고리: {list(market_context.keys())})")
except FileNotFoundError:
    print(f"오류: '{COMPETITOR_FILE_CLEANED}' 파일을 찾을 수 없습니다.")
    market_context = None

제품 15개 불러오기 완료.
페르소나 363개 불러오기 완료.
경쟁사 시장 데이터 불러오기 완료. (카테고리: ['RTD_액상커피', '그릭요거트', '참치액', '참치캔', '캔햄'])


## 2. 프롬프트 생성 함수 정의 (매핑 로직 최종 수정)

In [20]:
# [수정] 모든 제품 카테고리를 5개 대표 카테고리에 1:1로 직접 매핑합니다.
FINAL_CATEGORY_MAPPING = {
    '참치 > 참치캔 > 라이트스탠다드참치': '참치캔',
    '참치 > 참치캔 > 가미참치': '참치캔',
    '조미소스 > 조미료 > 액상조미료': '참치액',
    '우유류 > 발효유 > 호상-중대용량': '그릭요거트',
    '축산 > 햄/소시지 > 캔햄': '캔햄',
    '축산캔 > 고급축산캔 > 가미축산캔': '캔햄',
    '수산 > 수산캔 > 번데기/골뱅이/꽁치': '캔햄', # 캔햄 제품이 수산 카테고리에도 있을 수 있음
    '우유류 > 커피 > 커피-CUP': 'RTD_액상커피'
}

def get_main_category(detailed_category):
    """상세 카테고리명을 받아, 정의된 대표 카테고리명을 반환합니다."""
    # 먼저, 전체 이름이 일치하는지 확인합니다.
    if detailed_category in FINAL_CATEGORY_MAPPING:
        return FINAL_CATEGORY_MAPPING[detailed_category]
    # 만약 전체 이름이 없다면, 부분 일치하는 키워드가 있는지 확인합니다. (예비용)
    if '참치캔' in detailed_category: return '참치캔'
    if '참치액' in detailed_category or '액상조미료' in detailed_category: return '참치액'
    if '그릭' in detailed_category or '발효유' in detailed_category: return '그릭요거트'
    if '캔햄' in detailed_category: return '캔햄'
    if '커피' in detailed_category: return 'RTD_액상커피'
    return None

def create_single_prompt(product_info, persona_info, market_context):
    """제품, 페르소나, 시장 정보를 바탕으로 최종 프롬프트를 생성합니다."""
    product_str = json.dumps(product_info, ensure_ascii=False, indent=4)
    persona_str = json.dumps(persona_info, ensure_ascii=False, indent=4)

    main_category_key = get_main_category(product_info.get('category', ''))
    context_data = market_context.get(main_category_key, {})
    context_str = json.dumps(context_data, ensure_ascii=False, indent=4) if context_data else "{}"

    prompt_template = f"""# ROLE
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 \"제품 정보\", 원시 \"페르소나 데이터\", 그리고 \"시장 경쟁 환경\"을 종합적으로 분석하여, 페르소나가 해당 제품의 잠재 구매자로서 어떤 특징을 보일지 예측하고, 그 결과를 하나의 완결된 JSON 객체로 생성하는 것입니다.

# INSTRUCTION
아래의 모든 정보를 바탕으로, 페르소나가 해당 제품의 구매자로서 성립하는 **싱글턴 페르소나 JSON**을 생성하세요. 페르소나의 속성(attributes)과 제품의 특징(features)을 논리적으로 연결하여 구매 확률과 이유, 월별 구매 빈도를 예측해야 합니다.

# INPUT DATA
## 1. 제품 정보
{product_str}

## 2. 페르소나 데이터
{persona_str}

## 3. 시장 경쟁 환경
{context_str}

# OUTPUT FORMAT
반드시 아래와 같은 구조의 JSON 형식으로만 응답하세요. 다른 설명은 추가하지 마세요.

{{{{
  \"product_name\": \"{product_info.get('product_name', '')}\",
  \"persona_key\": {persona_info.get('persona_key')},
  \"purchase_behavior_prediction\": {{{{
    \"purchase_probability_pct\": \"<여기에 구매 확률(0-100)을 숫자로 예측>\",
    \"reason\": \"<여기에 페르소나 속성과 제품 특징, 시장 상황을 연결한 구매 결정 이유를 상세히 서술>\",
    \"monthly_purchase_frequency\": {{{{
      \"2024-07\": 0, \"2024-08\": 0, \"2024-09\": 0, \"2024-10\": 0, \"2024-11\": 0, \"2024-12\": 0,
      \"2025-01\": 0, \"2025-02\": 0, \"2025-03\": 0, \"2025-04\": 0, \"2025-05\": 0, \"2025-06\": 0
    }}}}
  }}}}
}}}} """
    return prompt_template.strip()

print("수정된 프롬프트 생성 함수가 정의되었습니다.")

수정된 프롬프트 생성 함수가 정의되었습니다.


## 3. 모든 조합에 대한 프롬프트 생성 및 저장

In [21]:
# [수정] 마지막 코드 셀 전체를 아래 코드로 교체하세요.

if 'market_context' in locals() and market_context is not None:
    print("프롬프트 생성을 시작합니다...")
    all_prompts = []
    missing_context_count = 0
    
    # [추가] 매칭에 실패한 제품을 저장할 리스트를 생성합니다.
    missing_context_products = []

    for product in products:
        # 프롬프트에 넣을 제품 정보 간소화
        product_essentials = {k: product.get(k) for k in ['brand', 'product_name', 'category', 'features', 'targeted_consumer', 'price_text', 'advertise_info']}

        # 시장 정보가 제대로 매칭되었는지 확인
        main_cat_key = get_main_category(product.get('category', ''))
        if not market_context.get(main_cat_key):
            missing_context_count += 1
            # [추가] 매칭 실패 시, 해당 제품 정보를 리스트에 추가합니다.
            missing_context_products.append(product)

        for persona in personas:
            prompt = create_single_prompt(product_essentials, persona, market_context)
            all_prompts.append({
                "product_name": product.get("product_name"),
                "persona_key": persona.get("persona_key"),
                "prompt": prompt
            })

    print(f"\n생성된 프롬프트를 '{OUTPUT_FILE}' 파일로 저장합니다...")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in all_prompts:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("="*40)
    print("🎉 모든 작업이 성공적으로 완료되었습니다!")
    print(f"총 {len(all_prompts)}개의 프롬프트가 생성되어 '{OUTPUT_FILE}'에 저장되었습니다.")
    
    if missing_context_count > 0:
        print(f"⚠️ 경고: 총 {len(products)}개 제품 중 {missing_context_count}개 제품의 시장 경쟁 환경 정보가 매칭되지 않았습니다.")
        print("   'FINAL_CATEGORY_MAPPING' 규칙을 확인하거나 추가해주세요.")
        
        # [추가] 매칭에 실패한 제품의 상세 정보를 출력합니다.
        print("\n--- [정보 누락 제품 목록] ---")
        for p in missing_context_products:
            print(f"  - 제품명: {p.get('product_name')}")
            print(f"    카테고리: {p.get('category')}")
        print("--------------------------")
else:
    print("시장 데이터(market_context)가 로드되지 않아 프롬프트 생성을 건너뜁니다.")

프롬프트 생성을 시작합니다...

생성된 프롬프트를 'prompts_for_llm.jsonl' 파일로 저장합니다...
🎉 모든 작업이 성공적으로 완료되었습니다!
총 5445개의 프롬프트가 생성되어 'prompts_for_llm.jsonl'에 저장되었습니다.
